# Normalización del dataset de Política

### 1. Importar librerias necesarias

In [56]:
import pandas as pd
import numpy as np
import unicodedata
import os

from pathlib import Path
from difflib import get_close_matches

### 2 - Carga del dataset a normalizar

In [57]:
# Definir la ruta del dataset correcto (archivo completo)
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\01 - raw\08 - politica\01 - alcaldes galicia.csv'

# Cargar el archivo CSV directamente
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

# Mostrar el número de municipios únicos
print(f"Municipios únicos en el dataset: {df['MUNICIPIO'].nunique()}")

Filas cargadas: 2101
Columnas disponibles: ['COMUNIDAD AUTÓNOMA', 'PROVINCIA', 'MUNICIPIO', 'PARTIDO POLITICO MUNICIPAL', 'FECHA POSESIÓN', 'FECHA BAJA']
Tamaño del dataset: 2101 filas x 6 columnas


,COMUNIDAD AUTÓNOMA,PROVINCIA,MUNICIPIO,PARTIDO POLITICO MUNICIPAL,FECHA POSESIÓN,FECHA BAJA
0,Galicia,Lugo ...,ABADIN,PP,03/07/1999,NaN
1,Galicia,Lugo ...,ABADÍN,PP,14/06/2003,NaN
2,Galicia,"Coruña, A ...",ABEGONDO,PP,03/07/1999,14/06/2003
3,Galicia,"Coruña, A ...",ABEGONDO,PP,14/06/2003,27/09/2004
4,Galicia,"Coruña, A ...",ABEGONDO,PP,14/06/2003,27/09/2004


Municipios únicos en el dataset: 689


In [58]:
# Exportar municipios originales a un txt para normalización manual
ruta_txt = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica'
import os
os.makedirs(ruta_txt, exist_ok=True)
archivo_municipios = os.path.join(ruta_txt, 'municipios originales a normalizar.txt')

# Detectar la columna de municipios automáticamente
col_municipio_detectada = None
if 'MUNICIPIO' in df.columns:
    col_municipio_detectada = 'MUNICIPIO'
elif 'municipio' in df.columns:
    col_municipio_detectada = 'municipio'
else:
    posibles_cols = [col for col in df.columns if 'municipio' in col.lower()]
    if posibles_cols:
        col_municipio_detectada = posibles_cols[0]

if col_municipio_detectada:
    municipios_originales = sorted(df[col_municipio_detectada].astype(str).unique())
    with open(archivo_municipios, 'w', encoding='utf-8') as f:
        for m in municipios_originales:
            f.write(m + '\n')
    print(f"Municipios originales exportados a: {archivo_municipios}")
    print(f"Total de municipios únicos encontrados: {len(municipios_originales)}")
    print(f"Columna utilizada: '{col_municipio_detectada}'")
else:
    print("No se encontró columna de municipios para exportar")
    print("Columnas disponibles:", list(df.columns))

Municipios originales exportados a: C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\municipios originales a normalizar.txt
Total de municipios únicos encontrados: 689
Columna utilizada: 'MUNICIPIO'


In [59]:
# Mostrar la primera fila como un diccionario columna: valor
primera_fila_dict = df.iloc[0].to_dict()
for k, v in primera_fila_dict.items():
    print(f"{k} = {v}")

COMUNIDAD AUTÓNOMA = Galicia
PROVINCIA = Lugo                                              
MUNICIPIO = ABADIN
PARTIDO POLITICO MUNICIPAL = PP
FECHA POSESIÓN = 03/07/1999
FECHA BAJA = nan


In [60]:
# Eliminar la columna 'geometry' (no lanza error si no existe)
df.drop(columns=['geometry'], inplace=True, errors='ignore')

In [61]:
# Eliminar la columna 'Unnamed: 0' (índice innecesario)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print("Columna 'Unnamed: 0' eliminada correctamente")
else:
    print("ℹLa columna 'Unnamed: 0' no existe en el dataset")

print(f"Columnas restantes: {list(df.columns)}")
print(f"Nuevas dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

ℹLa columna 'Unnamed: 0' no existe en el dataset
Columnas restantes: ['COMUNIDAD AUTÓNOMA', 'PROVINCIA', 'MUNICIPIO', 'PARTIDO POLITICO MUNICIPAL', 'FECHA POSESIÓN', 'FECHA BAJA']
Nuevas dimensiones: 2101 filas x 6 columnas


In [62]:
# Limpieza específica para el dataset de política
import unicodedata

print("LIMPIEZA ESPECÍFICA DE DATOS POLÍTICOS")
print("=" * 45)

# 1. Arreglar problemas de codificación UTF-8
def arreglar_codificacion(texto):
    if pd.isna(texto):
        return texto
    
    # Diccionario de reemplazos comunes de mal codificación
    reemplazos = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã±': 'ñ', 'Ã§': 'ç', 'Ãœ': 'Ü', 'Â': ''
    }
    
    texto_corregido = str(texto)
    for mal, bien in reemplazos.items():
        texto_corregido = texto_corregido.replace(mal, bien)
    
    return texto_corregido

# Aplicar corrección de codificación a la columna municipio
if 'municipio' in df.columns:
    print("Corrigiendo problemas de codificación en municipios...")
    df['municipio'] = df['municipio'].apply(arreglar_codificacion)
    print("Codificación corregida")

# 2. Arreglar formato "municipio a" -> "a municipio"
def arreglar_formato_municipio(nombre):
    if pd.isna(nombre):
        return nombre
    
    nombre = str(nombre).strip().lower()
    
    # Si termina en " a", moverlo al principio
    if nombre.endswith(' a'):
        nombre_base = nombre[:-2].strip()
        nombre = f"a {nombre_base}"
    
    return nombre

# Aplicar corrección de formato
if 'municipio' in df.columns:
    print("Corrigiendo formato de municipios (ej: 'arnoia a' -> 'a arnoia')...")
    municipios_antes = df['municipio'].unique()[:5]
    print(f"Ejemplos antes: {list(municipios_antes)}")
    
    df['municipio'] = df['municipio'].apply(arreglar_formato_municipio)
    
    municipios_despues = df['municipio'].unique()[:5] 
    print(f"Ejemplos después: {list(municipios_despues)}")
    print("Formato corregido")

# 3. Limpiar campo alcalde (quitar "Alcalde/sa de")
def limpiar_alcalde(nombre):
    if pd.isna(nombre):
        return nombre
    
    nombre = str(nombre).strip()
    
    # Remover prefijos comunes
    prefijos_remover = [
        'Alcalde/sa de ',
        'Alcalde de ',
        'Alcaldesa de ',
        'alcalde/sa de ',
        'alcalde de ',
        'alcaldesa de '
    ]
    
    for prefijo in prefijos_remover:
        if nombre.startswith(prefijo):
            municipio_en_nombre = nombre[len(prefijo):].strip()
            return f"Alcalde/sa de {municipio_en_nombre}"
    
    return nombre

# Aplicar limpieza al campo alcalde
if 'alcalde' in df.columns:
    print("Normalizando campo alcalde...")
    ejemplos_antes = df['alcalde'].dropna().head(3).tolist()
    print(f"Ejemplos antes: {ejemplos_antes}")
    
    df['alcalde'] = df['alcalde'].apply(limpiar_alcalde)
    
    ejemplos_despues = df['alcalde'].dropna().head(3).tolist()
    print(f"Ejemplos después: {ejemplos_despues}")
    print("Campo alcalde normalizado")

print(f"\nRESULTADO DE LA LIMPIEZA:")
print(f"Filas: {len(df)}")
print(f"Municipios únicos: {df['municipio'].nunique() if 'municipio' in df.columns else 'N/A'}")
if 'municipio' in df.columns:
    print(f"Primeros municipios: {sorted(df['municipio'].unique())[:10]}")

LIMPIEZA ESPECÍFICA DE DATOS POLÍTICOS

RESULTADO DE LA LIMPIEZA:
Filas: 2101
Municipios únicos: N/A


### 2.1 - Normalizar los nombres de las columnas

In [63]:
# Mostrar nombres originales de columnas
print('Nombres originales de columnas:')
print(list(df.columns))

# Normalizar nombres de columnas a español, minúsculas y descriptivos
# Para el dataset de política (alcaldes), mapeo específico según estructura real
columnas_renombrar = {
    'COMUNIDAD AUTÓNOMA': 'comunidad_autonoma',
    'PROVINCIA': 'provincia', 
    'MUNICIPIO': 'municipio',
    'PARTIDO POLITICO MUNICIPAL': 'partido_politico',
    'FECHA POSESIÓN': 'fecha_posesion',
    'FECHA BAJA': 'fecha_baja'
}

print('\nMapeo de nombres de columnas:')
for k, v in columnas_renombrar.items():
    if k in df.columns:
        print(f'{k} -> {v}')
    else:
        print(f'{k} -> {v} (columna no encontrada)')

# Renombrar columnas que existan
columnas_existentes = {k: v for k, v in columnas_renombrar.items() if k in df.columns}
if columnas_existentes:
    df.rename(columns=columnas_existentes, inplace=True)
    print(f'\nRenombradas {len(columnas_existentes)} columnas')
else:
    print('\nNo se encontraron columnas para renombrar')

print('\nNombres de columnas tras la normalización:')
print(list(df.columns))

# Verificar que tenemos las columnas esperadas para política
columnas_esperadas = ['comunidad_autonoma', 'provincia', 'municipio', 'partido_politico', 'fecha_posesion', 'fecha_baja']
columnas_presentes = [col for col in columnas_esperadas if col in df.columns]
columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]
columnas_extra = [col for col in df.columns if col not in columnas_esperadas]

print(f'\nVERIFICACIÓN DE COLUMNAS:')
print(f'Columnas esperadas presentes: {len(columnas_presentes)}/{len(columnas_esperadas)}')
print(f'Columnas presentes: {columnas_presentes}')
if columnas_faltantes:
    print(f'Columnas faltantes: {columnas_faltantes}')
if columnas_extra:
    print(f'ℹColumnas adicionales: {columnas_extra}')

Nombres originales de columnas:
['COMUNIDAD AUTÓNOMA', 'PROVINCIA', 'MUNICIPIO', 'PARTIDO POLITICO MUNICIPAL', 'FECHA POSESIÓN', 'FECHA BAJA']

Mapeo de nombres de columnas:
COMUNIDAD AUTÓNOMA -> comunidad_autonoma
PROVINCIA -> provincia
MUNICIPIO -> municipio
PARTIDO POLITICO MUNICIPAL -> partido_politico
FECHA POSESIÓN -> fecha_posesion
FECHA BAJA -> fecha_baja

Renombradas 6 columnas

Nombres de columnas tras la normalización:
['comunidad_autonoma', 'provincia', 'municipio', 'partido_politico', 'fecha_posesion', 'fecha_baja']

VERIFICACIÓN DE COLUMNAS:
Columnas esperadas presentes: 6/6
Columnas presentes: ['comunidad_autonoma', 'provincia', 'municipio', 'partido_politico', 'fecha_posesion', 'fecha_baja']


### 3 - Visualización y exploración inicial

In [64]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2101 entries, 0 to 2100
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   comunidad_autonoma  2101 non-null   object
 1   provincia           2101 non-null   object
 2   municipio           2101 non-null   object
 3   partido_politico    2101 non-null   object
 4   fecha_posesion      2101 non-null   object
 5   fecha_baja          1416 non-null   object
dtypes: object(6)
memory usage: 98.6+ KB


### 4 - Cargar el dataset limpio de municipios de Galicia

Vamos a cargar la tabla de municipios de Galicia que creamos antes. Usaremos estos nombres como referencia para normalizar los municipios del dataset de incendios.

In [65]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\01 - municipios\01 - municipios normalizados.csv'

# Cargar el archivo CSV de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


### 5 - Normalización automática de municipios (optimizada por mapeo único y orden invertido)

In [66]:
# Normalización automática de municipios siguiendo el patrón estándar
ruta_txt = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica'
os.makedirs(ruta_txt, exist_ok=True)

# Definir las columnas a usar
col_municipio = 'municipio'  # columna a normalizar en el dataset principal
col_ref = 'municipio'        # columna de referencia en el dataset de municipios
df_ref = df_municipios       # referencia oficial

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Crear diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# Obtener municipios únicos del dataset principal
municipios_originales_unicos = set(x for x in df[col_municipio].astype(str).unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

print(f"🔍 ANÁLISIS INICIAL:")
print(f"   - Municipios únicos en dataset político: {len(municipios_originales_unicos)}")
print(f"   - Municipios únicos en referencia: {len(municipios_referencia)}")

# Crear mapeo: municipio original -> municipio de referencia
mapeo = {}
pendientes = []
coincidencias_exactas = 0
coincidencias_similitud = 0
coincidencias_invertidas = 0

for municipio_original in municipios_originales_unicos:
    # Normalizar solo para la búsqueda
    clave_normalizada = normalizar_nombre(municipio_original)
    
    # 1. Buscar coincidencia exacta en referencia normalizada
    if clave_normalizada in ref_norm:
        mapeo[municipio_original] = ref_norm[clave_normalizada]
        coincidencias_exactas += 1
        continue
    
    # 2. Buscar coincidencia por similitud (cutoff 0.8)
    sugerencias = get_close_matches(clave_normalizada, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[municipio_original] = ref_norm[sugerencias[0]]
        coincidencias_similitud += 1
        continue
    
    # 3. Probar a invertir el orden de las palabras si hay exactamente dos
    partes = clave_normalizada.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[municipio_original] = ref_norm[invertido]
            coincidencias_invertidas += 1
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[municipio_original] = ref_norm[sugerencias_inv[0]]
            coincidencias_similitud += 1
            continue
    
    # Si no se encuentra nada, dejar como pendiente
    mapeo[municipio_original] = municipio_original  # Mantener original
    pendientes.append(municipio_original)

print(f"\nRESULTADOS DE NORMALIZACIÓN AUTOMÁTICA:")
print(f"   - Coincidencias exactas: {coincidencias_exactas}")
print(f"   - Coincidencias por similitud: {coincidencias_similitud}")
print(f"   - Coincidencias por inversión: {coincidencias_invertidas}")
print(f"   - Municipios pendientes: {len(pendientes)}")

# Aplicar correcciones manuales conocidas (municipios fusionados y casos especiales)
correcciones_manuales = {
    # Municipios fusionados históricos
    'Cesuras': 'oza-cesuras',
    'Oza dos Ríos': 'oza-cesuras',
    'cesuras': 'oza-cesuras',
    'oza dos rios': 'oza-cesuras',
    # Correcciones adicionales identificadas
    'CESURAS': 'oza-cesuras',
    'COVELO (O)': 'covelo',
    'OZA DOS RIOS': 'oza-cesuras',
    'OZA DOS RÍOS': 'oza-cesuras',
    'POBRA DE BROLLON (A)': 'a pobra do brollón',
    'SOBRADO DOS MONXES': 'sobrado'
}

for original, corregido in correcciones_manuales.items():
    if original in mapeo:
        mapeo[original] = corregido
        print(f"   ✓ Corrección manual aplicada: '{original}' -> '{corregido}'")

# Aplicar el mapeo a todo el dataset
df[col_municipio] = df[col_municipio].astype(str).map(mapeo).fillna(df[col_municipio])

# Diagnóstico final
municipios_normalizados = set(df[col_municipio].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"\nDIAGNÓSTICO FINAL:")
print(f"   - Municipios únicos normalizados: {len(municipios_normalizados)}")
print(f"   - Municipios de referencia no presentes: {len(faltan_en_dataset)}")
print(f"   - Municipios extra no en referencia: {len(sobran_en_dataset)}")

if sobran_en_dataset:
    print(f"   Municipios extra: {sorted(sobran_en_dataset)}")

# Exportar archivos de control
archivo_diccionario = os.path.join(ruta_txt, 'diccionario_normalizacion_final.txt')
with open(archivo_diccionario, 'w', encoding='utf-8', newline='') as f:
    import csv
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['original', 'normalizado'])
    for k, v in sorted(mapeo.items()):
        writer.writerow([k, v])

archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in sorted(pendientes):
        f.write(f'{m}\n')

print(f"\nARCHIVOS EXPORTADOS:")
print(f"   - Diccionario: {archivo_diccionario}")
print(f"   - Pendientes: {archivo_pendientes}")
print(f"\nNORMALIZACIÓN COMPLETADA")

🔍 ANÁLISIS INICIAL:
   - Municipios únicos en dataset político: 689
   - Municipios únicos en referencia: 315

RESULTADOS DE NORMALIZACIÓN AUTOMÁTICA:
   - Coincidencias exactas: 584
   - Coincidencias por similitud: 67
   - Coincidencias por inversión: 30
   - Municipios pendientes: 8
   ✓ Corrección manual aplicada: 'Cesuras' -> 'oza-cesuras'
   ✓ Corrección manual aplicada: 'Oza dos Ríos' -> 'oza-cesuras'
   ✓ Corrección manual aplicada: 'CESURAS' -> 'oza-cesuras'
   ✓ Corrección manual aplicada: 'COVELO (O)' -> 'covelo'
   ✓ Corrección manual aplicada: 'OZA DOS RIOS' -> 'oza-cesuras'
   ✓ Corrección manual aplicada: 'OZA DOS RÍOS' -> 'oza-cesuras'
   ✓ Corrección manual aplicada: 'POBRA DE BROLLON (A)' -> 'a pobra do brollón'
   ✓ Corrección manual aplicada: 'SOBRADO DOS MONXES' -> 'sobrado'

DIAGNÓSTICO FINAL:
   - Municipios únicos normalizados: 315
   - Municipios de referencia no presentes: 0
   - Municipios extra no en referencia: 0

ARCHIVOS EXPORTADOS:
   - Diccionario: C:\0

In [67]:
# Verificación final después de la normalización
print('VERIFICACIÓN FINAL DE MUNICIPIOS NORMALIZADOS:')
print("=" * 60)

municipios_finales = sorted(df['municipio'].unique())
print(f"Total de municipios únicos después de normalización: {len(municipios_finales)}")
print(f"Primeros 10 municipios: {municipios_finales[:10]}")
print(f"Últimos 10 municipios: {municipios_finales[-10:]}")

# Verificar que no hay valores nulos
nulos = df['municipio'].isnull().sum()
print(f"\nValores nulos en municipio: {nulos}")

# Comparar con la referencia oficial
municipios_referencia_final = set(df_municipios['municipio'].unique())
municipios_dataset_final = set(df['municipio'].unique())

coincidencias = len(municipios_referencia_final.intersection(municipios_dataset_final))
print(f"\nCoincidencias con referencia oficial: {coincidencias}/{len(municipios_referencia_final)} ({coincidencias/len(municipios_referencia_final)*100:.1f}%)")

print("Verificación completada.")

VERIFICACIÓN FINAL DE MUNICIPIOS NORMALIZADOS:
Total de municipios únicos después de normalización: 315
Primeros 10 municipios: ['a arnoia', 'a baña', 'a bola', 'a capela', 'a cañiza', 'a coruña', 'a estrada', 'a fonsagrada', 'a guarda', 'a gudiña']
Últimos 10 municipios: ['vilarmaior', 'vilasantar', 'vimianzo', 'viveiro', 'xermade', 'xinzo de limia', 'xove', 'xunqueira de ambía', 'xunqueira de espadanedo', 'zas']

Valores nulos en municipio: 0

Coincidencias con referencia oficial: 315/315 (100.0%)
Verificación completada.


### 6. Exportar el dataset final con municipios normalizados

A continuación se exporta el dataframe resultante, que contiene todos los municipios normalizados y desdoblados, a archivos Excel y CSV para su uso posterior.

In [68]:
# La columna 'municipio' ya contiene los valores normalizados, no es necesario sustituir ni eliminar auxiliares
print("Columna 'municipio' actualizada con los valores normalizados.")

Columna 'municipio' actualizada con los valores normalizados.


In [69]:
# Exportar el dataframe final con municipios normalizados
print("EXPORTANDO DATASET FINAL NORMALIZADO")
print("=" * 50)

ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - politica municipios normalizados.csv')

# Eliminar columnas duplicadas si las hubiera
df = df.loc[:, ~df.columns.duplicated()]

# Verificar estructura final antes de exportar
print(f"Estructura del dataset final:")
print(f"   - Filas: {len(df):,}")
print(f"   - Columnas: {len(df.columns)}")
print(f"   - Columnas: {list(df.columns)}")
print(f"   - Municipios únicos: {df['municipio'].nunique()}")

# Exportar a CSV
df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'\nDataset final exportado como: {archivo_export}')

# Mostrar muestra del resultado
print(f"\nMUESTRA DEL DATASET FINAL:")
display(df.head())

EXPORTANDO DATASET FINAL NORMALIZADO
Estructura del dataset final:
   - Filas: 2,101
   - Columnas: 6
   - Columnas: ['comunidad_autonoma', 'provincia', 'municipio', 'partido_politico', 'fecha_posesion', 'fecha_baja']
   - Municipios únicos: 315

Dataset final exportado como: C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\01 - politica municipios normalizados.csv

MUESTRA DEL DATASET FINAL:


,comunidad_autonoma,provincia,municipio,partido_politico,fecha_posesion,fecha_baja
0,Galicia,Lugo ...,abadín,PP,03/07/1999,NaN
1,Galicia,Lugo ...,abadín,PP,14/06/2003,NaN
2,Galicia,"Coruña, A ...",abegondo,PP,03/07/1999,14/06/2003
3,Galicia,"Coruña, A ...",abegondo,PP,14/06/2003,27/09/2004
4,Galicia,"Coruña, A ...",abegondo,PP,14/06/2003,27/09/2004


In [70]:
# Comprobación final: verificar calidad de la normalización
print("COMPROBACIÓN FINAL DE CALIDAD")
print("=" * 50)

csv_exportado = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\01 - politica municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)

municipios_exportados = set(df_exportado['municipio'].dropna().unique())
municipios_referencia = set(df_municipios['municipio'].dropna().unique())

print(f"ESTADÍSTICAS FINALES:")
print(f"   - Municipios únicos en CSV exportado: {len(municipios_exportados)}")
print(f"   - Municipios únicos en referencia oficial: {len(municipios_referencia)}")
print(f"   - Registros totales en dataset: {len(df_exportado):,}")

# Calcular coincidencias
coincidencias = municipios_exportados.intersection(municipios_referencia)
print(f"   - Coincidencias exactas: {len(coincidencias)}")
print(f"   - Porcentaje de cobertura: {len(coincidencias)/len(municipios_referencia)*100:.1f}%")

# Identificar diferencias
faltan_en_exportado = municipios_referencia - municipios_exportados
sobran_en_exportado = municipios_exportados - municipios_referencia

if len(municipios_exportados) == len(municipios_referencia) and len(sobran_en_exportado) == 0:
    print('\n¡PERFECTO! El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!')
else:
    if faltan_en_exportado:
        print(f'\nMunicipios de referencia que faltan ({len(faltan_en_exportado)}): {sorted(faltan_en_exportado)}')
    if sobran_en_exportado:
        print(f'\nMunicipios extra no en referencia ({len(sobran_en_exportado)}): {sorted(sobran_en_exportado)}')

print(f"\nRESULTADO: {'EXCELENTE' if len(sobran_en_exportado) == 0 else 'BUENO - Revisar municipios extra'}")

COMPROBACIÓN FINAL DE CALIDAD
ESTADÍSTICAS FINALES:
   - Municipios únicos en CSV exportado: 315
   - Municipios únicos en referencia oficial: 315
   - Registros totales en dataset: 2,101
   - Coincidencias exactas: 315
   - Porcentaje de cobertura: 100.0%

¡PERFECTO! El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!

RESULTADO: EXCELENTE


In [71]:
# Aplicar correcciones adicionales identificadas en la comprobación
print("APLICANDO CORRECCIONES ADICIONALES")
print("=" * 50)

# Correcciones adicionales basadas en el análisis
correcciones_adicionales = {
    'CESURAS': 'oza-cesuras',
    'COVELO (O)': 'covelo',
    'OZA DOS RIOS': 'oza-cesuras',
    'OZA DOS RÍOS': 'oza-cesuras',
    'POBRA DE BROLLON (A)': 'a pobra do brollón',
    'SOBRADO DOS MONXES': 'sobrado'
}

# Aplicar las correcciones al dataset en memoria
correcciones_aplicadas = 0
for original, corregido in correcciones_adicionales.items():
    if original in df['municipio'].values:
        df.loc[df['municipio'] == original, 'municipio'] = corregido
        correcciones_aplicadas += 1
        print(f"   ✓ '{original}' → '{corregido}'")

print(f"\nRESULTADOS:")
print(f"   - Correcciones aplicadas: {correcciones_aplicadas}")
print(f"   - Municipios únicos después de correcciones: {df['municipio'].nunique()}")

# Re-exportar el dataset corregido SOLO al archivo válido
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica'
archivo_export = os.path.join(ruta_export, '01 - politica municipios normalizados.csv')

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f"\nDataset corregido re-exportado como: {archivo_export}")

print("Correcciones adicionales aplicadas.")

APLICANDO CORRECCIONES ADICIONALES

RESULTADOS:
   - Correcciones aplicadas: 0
   - Municipios únicos después de correcciones: 315

Dataset corregido re-exportado como: C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\01 - politica municipios normalizados.csv
Correcciones adicionales aplicadas.


In [72]:
# Verificación final después de aplicar correcciones adicionales
print("VERIFICACIÓN FINAL DESPUÉS DE CORRECCIONES")
print("=" * 60)

# Cargar el dataset corregido para verificar
csv_corregido = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\01 - politica municipios normalizados.csv'
df_final_verificacion = pd.read_csv(csv_corregido)

municipios_finales_verificacion = set(df_final_verificacion['municipio'].dropna().unique())
municipios_referencia_verificacion = set(df_municipios['municipio'].dropna().unique())

print(f"ESTADÍSTICAS FINALES CORREGIDAS:")
print(f"   - Municipios únicos en CSV final: {len(municipios_finales_verificacion)}")
print(f"   - Municipios únicos en referencia oficial: {len(municipios_referencia_verificacion)}")
print(f"   - Registros totales en dataset: {len(df_final_verificacion):,}")

# Calcular coincidencias finales
coincidencias_finales = municipios_finales_verificacion.intersection(municipios_referencia_verificacion)
print(f"   - Coincidencias exactas: {len(coincidencias_finales)}")
print(f"   - Porcentaje de cobertura: {len(coincidencias_finales)/len(municipios_referencia_verificacion)*100:.1f}%")

# Verificar si aún hay municipios extra
sobran_finales = municipios_finales_verificacion - municipios_referencia_verificacion
faltan_finales = municipios_referencia_verificacion - municipios_finales_verificacion

if len(sobran_finales) == 0 and len(faltan_finales) == 0:
    print('\n¡PERFECTO! El dataset final tiene exactamente los mismos municipios que la referencia oficial!')
    resultado_final = "EXCELENTE"
else:
    if faltan_finales:
        print(f'\nMunicipios de referencia que aún faltan ({len(faltan_finales)}): {sorted(faltan_finales)}')
    if sobran_finales:
        print(f'\nMunicipios extra que aún sobran ({len(sobran_finales)}): {sorted(sobran_finales)}')
    resultado_final = "BUENO - Revisar municipios restantes"

print(f"\nRESULTADO FINAL: {resultado_final}")
print(f"Proceso de normalización completado.")

# Exportar lista final de municipios para verificación
archivo_municipios_finales_corregidos = os.path.join(ruta_export, 'municipios dataset final corregido.txt')
municipios_lista_final = sorted(df_final_verificacion['municipio'].dropna().unique())
with open(archivo_municipios_finales_corregidos, 'w', encoding='utf-8') as f:
    for m in municipios_lista_final:
        f.write(m + '\n')
print(f"\nLista final exportada a: {archivo_municipios_finales_corregidos}")

VERIFICACIÓN FINAL DESPUÉS DE CORRECCIONES
ESTADÍSTICAS FINALES CORREGIDAS:
   - Municipios únicos en CSV final: 315
   - Municipios únicos en referencia oficial: 315
   - Registros totales en dataset: 2,101
   - Coincidencias exactas: 315
   - Porcentaje de cobertura: 100.0%

¡PERFECTO! El dataset final tiene exactamente los mismos municipios que la referencia oficial!

RESULTADO FINAL: EXCELENTE
Proceso de normalización completado.

Lista final exportada a: C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\municipios dataset final corregido.txt


In [73]:
# Exportar los municipios únicos del dataset final a un txt
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica'
os.makedirs(ruta_export, exist_ok=True)
archivo_municipios_finales = os.path.join(ruta_export, 'municipios dataset final.txt')
municipios_finales = sorted(df['municipio'].dropna().unique())
with open(archivo_municipios_finales, 'w', encoding='utf-8') as f:
    for m in municipios_finales:
        f.write(m + '\n')
print(f"Archivo exportado con los municipios finales: {archivo_municipios_finales}")
print(f"Total de municipios únicos en el dataset final: {len(municipios_finales)}")

Archivo exportado con los municipios finales: C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\municipios dataset final.txt
Total de municipios únicos en el dataset final: 315
